# 02 · 模型训练（Colab）

训练 `crn-nano` / `crn-lite` / `crn-large` 三档模型。

**为什么是三档而不是一个**：指标表里最有说服力的不是"我的模型 SI-SDR 多少"，
而是**帕累托前沿** —— 参数量/RTF 与质量的权衡曲线。只有一个点画不出曲线，
也就回答不了"要不要为 0.3 dB 多花 3 倍算力"这类真实的工程问题。

**训练数据是 DNS 英文语音 + DNS 真实噪声 + DNS 真实 RIR**，在线随机混音。
评测则在中文 WenetSpeech 上做 —— 跨语种，见 01 的说明。

训练产物全部保存到 Drive（`checkpoints/<模型名>/`）：
`last.pt`（每 epoch 覆盖，用于续训）、`best.pt`（验证最优，导出用它）、
`history.json`（训练曲线）。Colab 会话随时可能断，checkpoint 里存了
**优化器动量、学习率调度、随机数状态**，断线后重跑训练 cell 会自动续训，
不会出现 loss 反弹。

In [ ]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与开关集中在这一个 cell，别处不要再写死路径
# ═══════════════════════════════════════════════════════════════════════

# Drive 上的项目根。**持久**，会话结束不丢。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 语料怎么放 ─────────────────────────────────────────────────────────
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每会话解压到本地盘。
#               一次下载永久有效；训练读取走本地盘全速。
#   'local'  —— 全在临时盘，用完即删，每个新会话都要重下。
DATA_MODE = 'hybrid'

# ── 数据规模 ───────────────────────────────────────────────────────────
# DNS5 干净语音的 split 切片数。每片 5.24 GB，实测约 **19 小时**，
# 落在"20~30 小时可管理子集"这个目标区间内。
# 切片档解压到末尾会报 EOF，属正常（详见 fetch_dns 的说明）。
N_SPEECH_SHARDS = 1
# DNS 噪声分片：audioset（日常环境声）+ freesound（标注音效）各取几片。
N_AUDIOSET_SHARDS = 2
N_FREESOUND_SHARDS = 1

# ── 快速验证模式 ───────────────────────────────────────────────────────
# True = 跳过 DNS 大文件，只下 WenetSpeech（约 520 MB）验证整条链路。
QUICK_TEST = False

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都靠它

assert DATA_MODE in ('hybrid', 'local'), 'DATA_MODE 只能是 hybrid / local'

DRIVE = DRIVE_ROOT
WORK = WORK_ROOT
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'

CKPT_DIR = f'{DRIVE}/checkpoints'   # 训练断点，每 epoch 保存
MODEL_DIR = f'{DRIVE}/models'       # 导出的 ONNX
TESTSET_DIR = f'{DRIVE}/testset'    # 固定测试集
LOG_DIR = f'{DRIVE}/logs'

assert os.path.isdir(DRIVE), (
    f'Drive 上找不到 {DRIVE}\n'
    '检查：① Drive 已挂载成功；② DRIVE_ROOT 与你实际的目录一致（区分大小写，空格照写）。'
)
for d in [WORK, ARCHIVE_DIR, DATA, CKPT_DIR, MODEL_DIR, TESTSET_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局')
print('─' * 74)
print(f'  代码包(需手动上传)  {DRIVE}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}')
print(f'  语料解压目标        {DATA}')
print(f'  数据清单            {DRIVE}/manifest.json')
print(f'  固定测试集          {TESTSET_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}')
print(f'  语料      {"快速验证(仅 WenetSpeech)" if QUICK_TEST else "完整(DNS 英文训练 + WenetSpeech 中文评测)"}')
print()

!df -h /content | tail -1

In [ ]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 DRIVE_ROOT 目录下。**代码改过就要重新上传**，
# 否则 Colab 跑的还是旧逻辑（这个坑踩过，见 docs/ISSUES.md）。
ZIP = f'{DRIVE}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/rtse-colab.zip 上传到 Drive 的 {DRIVE} 下。\n'
    f'该目录下现有：{sorted(os.listdir(DRIVE))[:12]}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没预装的。不用 `pip install -e .`：那会去解析 pyproject
# 里锁定的 torch CPU 索引，把 Colab 自带的 GPU 版 torch 覆盖掉，训练慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime opencc-python-reimplemented 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
import rtse
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── 自检：Colab 侧与本地必须是同一条信号链路 ───────────────────────────
# 这一步不能跳。Colab 上的 STFT 与本地哪怕差一点，训练出来的模型拿回本地就会
# 掉点，而且极难定位（两边单独看都"没问题"）。
import numpy as np, torch
from rtse.audio.stft import stft, istft, check_cola, magnitude_db
from rtse.data.dataset import stft_torch, istft_torch

x = np.random.default_rng(0).standard_normal(16000)
print('COLA 偏差          :', f'{check_cola():.2e}')
print('numpy 完美重构      :', f'{np.max(np.abs(istft(stft(x), length=x.size) - x)):.2e}')

xt = torch.from_numpy(x).float().unsqueeze(0)
ref, got = stft(x), stft_torch(xt)
assert ref.shape[0] == got.shape[2], f'帧数不一致 {ref.shape[0]} vs {got.shape[2]}'
gc = got[0,0].numpy() + 1j*got[0,1].numpy()
print('torch/numpy STFT   :', f'{np.max(np.abs(gc - ref)) / np.max(np.abs(ref)):.2e} (相对)')
print('torch 往返重构      :', f'{(istft_torch(stft_torch(xt), length=16000) - xt).abs().max().item():.2e}')

t = np.arange(16000)/16000
db = magnitude_db(stft(np.sin(2*np.pi*1000*t))).max()
print('dBFS 标定(满幅正弦) :', f'{db:.3f} dB  (应为 0.000)')
assert abs(db) < 0.05, 'dBFS 标定不对，检查代码包是否为最新'
print('\n✅ Colab 与本地是同一条链路。')

## 1. 构建数据集

Colab 每次连接给的都是**全新虚拟机**，本地盘是空的 —— 哪怕几分钟前刚在 01 里
跑完下载。下面会用 Drive 上缓存的压缩包重新解压（几分钟），压缩包也不在了才重下。
**这是正常现象，不用手动跳回 01。**

In [ ]:
mf = Path(f'{DRIVE}/manifest.json')
assert mf.exists(), f'找不到 {mf}，先跑 01_data_prep.ipynb'
manifest = json.loads(mf.read_text(encoding='utf-8'))
QUICK_TEST = manifest['quick_test']
print(f'清单版本: {manifest.get("version")}   quick_test={QUICK_TEST}')
print(f'  语音 train/val: {len(manifest["speech"]["train"])}/{len(manifest["speech"]["val"])}')
print(f'  噪声 稳态/非稳态(train): {len(manifest["noise_stationary_train"])}/'
      f'{len(manifest["noise_nonstationary_train"])}')
print(f'  真实 RIR train: {len(manifest["rir_train"])}')

In [ ]:
DNS_BASE = 'https://dnschallengepublic.blob.core.windows.net/dns5archive/V5_training_dataset'

def fetch_dns(name, blob_path, expect_min_wavs=50, partial_ok=False):
    """下载 → 解压一个 DNS 分片。带下载/解压双标记，支持断点续传与跨会话复用。

    Args:
        partial_ok: 该分片是 `split` 切片（干净语音），解压到末尾必然报
            "Unexpected EOF"。设 True 时忽略这个错误——只要解出足够多的
            完整文件就算成功。校验靠**实际解出的 wav 数量**，不靠 tar 的返回码。

    校验方式统一是"解压后递归扫到的 wav 数量"，而不是断言某个具体子目录名——
    实测 DNS5 语音解出来的路径是 `mnt/dnsv5/clean/read_speech/...`，
    嵌套好几层且没写在官方文档里。下游 scan() 本来就是递归扫描，不关心层数。
    """
    dl_mark = f'{ARCHIVE_DIR}/.{name}.downloaded'
    ex_mark = f'{DATA}/.{name}.extracted'
    fname = blob_path.rsplit('/', 1)[-1]
    archive = f'{ARCHIVE_DIR}/{fname}'
    out_dir = f'{DATA}/{name}'
    os.makedirs(out_dir, exist_ok=True)

    def count_wavs():
        r = subprocess.run(f'find {shq(out_dir)} -name "*.wav" | wc -l',
                           shell=True, capture_output=True, text=True)
        return int((r.stdout or '0').strip() or 0)

    if os.path.exists(ex_mark) and count_wavs() >= expect_min_wavs:
        print(f'[skip  ] {name} 已解压（{count_wavs()} 个 wav）'); return True

    if os.path.exists(dl_mark) and os.path.exists(archive):
        print(f'[cached] {name} 压缩包已在 Drive ({os.path.getsize(archive)/1e9:.2f} GB)')
    else:
        url = f'{DNS_BASE}/{blob_path}'
        print(f'[get   ] {name} ← {url}')
        rc = os.system(f'wget -q --show-progress -c -T 60 -O {shq(archive)} {shq(url)}')
        if rc != 0 or not os.path.exists(archive) or os.path.getsize(archive) < 1e6:
            print(f'[FAIL  ] {name} 下载失败。去 https://github.com/microsoft/DNS-Challenge '
                  f'确认 blob 路径是否变了。')
            return False
        Path(dl_mark).touch()

    print(f'[unpack] {name}  ({os.path.getsize(archive)/1e9:.2f} GB) → {out_dir}')
    t = time.time()
    flag = 'xzf' if archive.endswith(('.tgz', '.tar.gz')) else 'xjf'
    # 切片档忽略 tar 的非零返回码（末尾必然 EOF），靠文件数判断成败
    os.system(f'tar -{flag} {shq(archive)} -C {shq(out_dir)} 2>/dev/null'
              + (' || true' if partial_ok else ''))
    n = count_wavs()
    if n < expect_min_wavs:
        print(f'[FAIL  ] 解压后只找到 {n} 个 wav，压缩包可能不完整。'
              f'删掉 {archive} 和 {dl_mark} 后重跑本 cell')
        return False

    if not KEEP_ARCHIVE:
        os.remove(archive); Path(dl_mark).unlink(missing_ok=True)
    Path(ex_mark).touch()
    note = '（切片档，尾部 EOF 属正常）' if partial_ok else ''
    print(f'[done  ] {name}   {n} 个 wav   解压耗时 {time.time()-t:.0f} 秒 {note}')
    return True


def dns_shards():
    """按配置生成要下载的分片清单 (名字, blob 路径)。"""
    # 语音是 split 切片，按字母序 partaa/partab/...；每片约 5.24 GB ≈ 19 小时
    parts = ['aa', 'ab', 'ac', 'ad', 'ae']
    sp = [(f'dns_speech_{p}', f'Track1_Headset/read_speech.tgz.part{p}')
          for p in parts[:N_SPEECH_SHARDS]]
    nz = [(f'dns_noise_audioset_{i:03d}',
           f'noise_fullband/datasets_fullband.noise_fullband.audioset_{i:03d}.tar.bz2')
          for i in range(N_AUDIOSET_SHARDS)]
    nz += [(f'dns_noise_freesound_{i:03d}',
            f'noise_fullband/datasets_fullband.noise_fullband.freesound_{i:03d}.tar.bz2')
           for i in range(N_FREESOUND_SHARDS)]
    # ⚠️ IR 分片在 blob 根目录下，**没有** `impulse_responses/` 前缀
    # （语音和噪声分片才有目录前缀）。写错会 404 —— 本地 HEAD 请求实测确认过。
    ir = [('dns_ir', 'datasets_fullband.impulse_responses_000.tar.bz2')]
    return sp, nz, ir

# 语料解压在临时盘，新会话要重来一遍。复用 01 里同一个 fetch_dns()。
sp_shards, nz_shards, ir_shards = dns_shards()
if not QUICK_TEST:
    ok = {}
    for n, b in sp_shards: ok[n] = fetch_dns(n, b, expect_min_wavs=500, partial_ok=True)
    for n, b in nz_shards: ok[n] = fetch_dns(n, b, expect_min_wavs=100)
    for n, b in ir_shards: ok[n] = fetch_dns(n, b, expect_min_wavs=50)
    assert all(ok.values()), '有语料没能自动补齐，看上面的 FAIL 信息'
print('语料就绪。')

In [ ]:
from torch.utils.data import DataLoader
from rtse.data.dataset import OnlineMixDataset, MixConfig
from rtse.metrics.intrusive import si_sdr

# OnlineMixDataset 接受**文件列表**。必须用清单而不是让它扫目录 ——
# 清单是按说话人划分好的，扫目录会把验证说话人混进训练集，指标全部虚高。
MIX = MixConfig(segment_seconds=4.0, snr_range=(-5.0, 20.0), reverb_prob=0.5)

train_ds = OnlineMixDataset(manifest['speech']['train'], manifest['noise_train'],
                            manifest['rir_train'], cfg=MIX, length=20000, seed=0)
val_ds = OnlineMixDataset(manifest['speech']['val'], manifest['noise_test'],
                          manifest['rir_test'], cfg=MIX, length=800, seed=999)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2,
                      pin_memory=True, drop_last=True, persistent_workers=True)
val_dl = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

nb_, cl_ = next(iter(train_dl))
print('batch', tuple(nb_.shape), '| 输入 SI-SDR 抽样:',
      [round(si_sdr(cl_[i].numpy(), nb_[i].numpy()), 1) for i in range(4)], 'dB')

## 2. 训练

**别照搬预估时间，看第一个 epoch 的实测值。** 训练器把每个 epoch 的
`epoch_seconds` 写进 `history.json`，第一个 epoch 跑完就能算出总时长。

第一次建议把 `EPOCHS` 改成 5 跑一轮，确认 loss 在降、时间可接受，再改回 60。
先跑 `crn-nano`（参数量只有 lite 的 1/5），它跑完就能拿到一套完整的端到端指标，
把整个流程闭环；大模型再慢慢跑。**有一个能用的模型，远好过两个都卡在半路。**

In [ ]:
from rtse.models import build_model
from rtse.train import Trainer, TrainConfig

MODELS = ['crn-nano', 'crn-lite']      # 想跑大模型就加上 'crn-large'
EPOCHS = 60

for name in MODELS:
    out_dir = f'{CKPT_DIR}/{name}'     # ← 落在 Drive 上，会话断了也在
    os.makedirs(out_dir, exist_ok=True)
    cfg = TrainConfig(model=name, epochs=EPOCHS, batch_size=16, lr=3e-4,
                      out_dir=out_dir, num_workers=2, log_every=100)
    model = build_model(name)
    print(f'\n{"="*70}\n{name}   参数量 {model.count_params():,}   → {out_dir}\n{"="*70}')

    tr = Trainer(model, train_dl, val_dl, cfg)
    last = Path(out_dir) / 'last.pt'
    if last.exists():
        tr.load(last)                  # 断点续训（含优化器/调度/随机数状态）
    if tr.epoch >= EPOCHS:
        print(f'{name} 已完成（epoch {tr.epoch}），跳过'); continue
    tr.fit()

print('\n训练产物（在 Drive 上）：')
!ls -lh {shq(CKPT_DIR)}/*/

## 3. 训练曲线

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name in MODELS:
    h = Path(f'{CKPT_DIR}/{name}/history.json')
    if not h.exists(): continue
    hist = json.loads(h.read_text())
    ep = [r['epoch'] for r in hist]
    axes[0].plot(ep, [r.get('loss') for r in hist], label=name)
    axes[1].plot(ep, [r.get('si_sdr') for r in hist], label=f'{name} train')
    if 'val_si_sdr' in hist[0]:
        axes[1].plot(ep, [r.get('val_si_sdr') for r in hist], '--', label=f'{name} val')
    axes[2].plot(ep, [r.get('spec') for r in hist], label=name)

for ax, t in zip(axes, ['总损失', 'SI-SDR (dB)', '压缩谱损失']):
    ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# 怎么读：
#   训练 SI-SDR 一路涨但验证走平 → 过拟合，加数据或加正则
#   两条都走平且数值低         → 欠拟合，或学习率有问题